In [1]:
import numpy as np
import polars as pl

from dataset.ohlcv.schema import OHLCV, OHLCV_PLDF


def generate_random_ohlcv(num_rows: int = 10_0000) -> OHLCV:
    """ランダムなOHLCVデータを生成する関数"""

    # ベース日時の生成（1分足を想定）
    start_date = pl.datetime(2000, 1, 1)  # ←引数を個別に指定
    base_date = pl.datetime_range(
        start=start_date,
        end=start_date + pl.duration(minutes=num_rows-1),
        interval="1m",
        eager=True
    ).alias("Date")

    # ランダムな価格変動（幾何ブラウン運動を模倣）
    returns = np.random.normal(0, 0.0001, num_rows)
    close_prices = 100.0 * np.exp(np.cumsum(returns))

    return pl.DataFrame({
        "Date": base_date,
        "Open": close_prices * np.random.uniform(0.99, 1.01, num_rows),
        "High": close_prices * np.random.uniform(1.0, 1.02, num_rows),
        "Low": close_prices * np.random.uniform(0.98, 1.0, num_rows),
        "Close": close_prices,
        "Volume": np.random.normal(1_000_000, 100_000, num_rows).astype(np.int64)
    }).cast(OHLCV_PLDF.schema)


In [2]:
ohlcv = generate_random_ohlcv()
print(ohlcv)
print(ohlcv.shape)
# 生成したOHLCVデータをParquet形式で保存
# ohlcv.write_parquet("ohlcv_sample.parq")
# MEMO: 1b=35gb, 100m=3.5gb

shape: (100_000, 6)
┌─────────────────────┬────────────┬────────────┬───────────┬────────────┬─────────┐
│ Date                ┆ Open       ┆ High       ┆ Low       ┆ Close      ┆ Volume  │
│ ---                 ┆ ---        ┆ ---        ┆ ---       ┆ ---        ┆ ---     │
│ datetime[μs]        ┆ f64        ┆ f64        ┆ f64       ┆ f64        ┆ i64     │
╞═════════════════════╪════════════╪════════════╪═══════════╪════════════╪═════════╡
│ 2000-01-01 00:00:00 ┆ 99.670281  ┆ 101.477571 ┆ 98.291187 ┆ 99.993751  ┆ 1015179 │
│ 2000-01-01 00:01:00 ┆ 99.748542  ┆ 99.985229  ┆ 99.742889 ┆ 99.984828  ┆ 1013644 │
│ 2000-01-01 00:02:00 ┆ 100.309952 ┆ 101.401411 ┆ 99.237371 ┆ 99.992189  ┆ 1022103 │
│ 2000-01-01 00:03:00 ┆ 100.434604 ┆ 101.592599 ┆ 98.879787 ┆ 99.996432  ┆ 1081429 │
│ 2000-01-01 00:04:00 ┆ 100.850083 ┆ 101.256788 ┆ 99.186923 ┆ 100.003958 ┆ 1023415 │
│ …                   ┆ …          ┆ …          ┆ …         ┆ …          ┆ …       │
│ 2000-03-10 10:35:00 ┆ 98.317378  ┆ 99.22511

In [3]:
# Close価格の時系列プロット
import plotly.express as px

fig = px.line(ohlcv, x="Date", y="Close", title="Close Price Over Time")
fig.show()